<a href="https://colab.research.google.com/github/2403a52233-sudo/NLP_/blob/main/Lab12_2_TextCNN_MultiFilter_T_Ganesh_2403A52233.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Import Libraries

In [2]:
# Numerical operations
import numpy as np

# Keras modules
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Install Gensim for Word2Vec
!pip install gensim

# Gensim for Word2Vec
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.4 MB/s eta 0:00:00


 Load and Explore Dataset

In [3]:
texts = [
    "this movie is awful",
    "i did not like this film",
    "the story was boring",
    "terrible acting",
    "waste of time",
    "very disappointing experience",
]

# Labels: 1 = positive, 0 = negative
labels = [1,1,0,0,1,0]

Text Preprocessing

In [4]:
tokenizer = Tokenizer()

# Learn vocabulary
tokenizer.fit_on_texts(texts)

# Convert sentences into sequences
sequences = tokenizer.texts_to_sequences(texts)

# Vocabulary dictionary
word_index = tokenizer.word_index

print(word_index)

{'this': 1, 'movie': 2, 'is': 3, 'awful': 4, 'i': 5, 'did': 6, 'not': 7, 'like': 8, 'film': 9, 'the': 10, 'story': 11, 'was': 12, 'boring': 13, 'terrible': 14, 'acting': 15, 'waste': 16, 'of': 17, 'time': 18, 'very': 19, 'disappointing': 20, 'experience': 21}


In [5]:
print(sequences)

[[1, 2, 3, 4], [5, 6, 7, 8, 1, 9], [10, 11, 12, 13], [14, 15], [16, 17, 18], [19, 20, 21]]


Vocabulary and Encoding

In [6]:
max_length = 6

X = pad_sequences(sequences, maxlen=max_length)

y = np.array(labels)

In [7]:
X

array([[ 0,  0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  1,  9],
       [ 0,  0, 10, 11, 12, 13],
       [ 0,  0,  0,  0, 14, 15],
       [ 0,  0,  0, 16, 17, 18],
       [ 0,  0,  0, 19, 20, 21]], dtype=int32)

In [8]:
sentences = [text.split() for text in texts]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1
)

In [9]:
vocab_size = len(word_index) + 1
embedding_dim = 50

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, index in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[index] = w2v_model.wv[word]

In [10]:
embedding_matrix[1]

array([-1.07245450e-03,  4.72862710e-04,  1.02066994e-02,  1.80185456e-02,
       -1.86058991e-02, -1.42336180e-02,  1.29177449e-02,  1.79459769e-02,
       -1.00308564e-02, -7.52674323e-03,  1.47610093e-02, -3.06694279e-03,
       -9.07322671e-03,  1.31081035e-02, -9.72032081e-03, -3.63203534e-03,
        5.75315952e-03,  1.98374758e-03, -1.65704302e-02, -1.88976359e-02,
        1.46235321e-02,  1.01405242e-02,  1.35153867e-02,  1.52573106e-03,
        1.27017805e-02, -6.81073172e-03, -1.89280277e-03,  1.15371468e-02,
       -1.50432754e-02, -7.87220709e-03, -1.50231645e-02, -1.86008448e-03,
        1.90762375e-02, -1.46383336e-02, -4.66753729e-03, -3.87548213e-03,
        1.61548741e-02, -1.18617918e-02,  9.03248801e-05, -9.50746797e-03,
       -1.92071013e-02,  1.00145862e-02, -1.75191704e-02, -8.78365058e-03,
       -7.01999670e-05, -5.92362892e-04, -1.53224804e-02,  1.92294866e-02,
        9.96411592e-03,  1.84662864e-02])

In [11]:
input_layer = Input(shape=(max_length,))

In [12]:
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=max_length,
    trainable=False
)(input_layer)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [13]:
conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)

conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)

conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)

In [14]:
pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)

In [15]:
merged = Concatenate()([pool1, pool2, pool3])

In [16]:
dense = Dense(10, activation='relu')(merged)

In [17]:
output = Dense(1, activation='sigmoid')(dense)

In [18]:
model = Model(inputs=input_layer, outputs=output)

In [19]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [20]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 6, 50)     │      1,100 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 4, 100)    │     15,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 3, 100)    │     20,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 2, 100)    │     25,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 300)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10)        │      3,010 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         11 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 64,421 (251.64 KB)

 Trainable params: 63,321 (247.35 KB)

 Non-trainable params: 1,100 (4.30 KB)

In [21]:
model.fit(
    X,
    y,
    epochs=10,
    batch_size=2
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.5000 - loss: 0.6917
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8333 - loss: 0.6828
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 1.0000 - loss: 0.6723
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.6621
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 1.0000 - loss: 0.6530 
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 1.0000 - loss: 0.6427
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 1.0000 - loss: 0.6318 
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 1.0000 - loss: 0.6209
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 0.6089
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.5996


In [23]:
test_text = ["this movie is awful"]

seq = tokenizer.texts_to_sequences(test_text)

pad = pad_sequences(seq, maxlen=max_length)

prediction = model.predict(pad)

print(prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
[[0.5845892]]
